# Redis Transactions

## Overview
Redis provides atomicity through three mechanisms, each suited to different use cases:

| Mechanism | Atomicity | Read mid-transaction | Retry on conflict |
|---|---|---|---|
| **MULTI/EXEC** | Yes — queue + execute | No (values are `QUEUED`) | No built-in |
| **WATCH + MULTI/EXEC** | Yes — with conflict detection | Read before MULTI | Yes — retry loop |
| **Lua scripts** | Yes — single-threaded execution | Yes — full read-modify-write | Yes — script decides |

**Key insight:** Redis does **not** have SQL-style transactions with rollback. MULTI/EXEC guarantees that all commands execute atomically — but if one command fails, the rest still run.

```python
# redis-py: pipeline(transaction=True) wraps commands in MULTI/EXEC
pipe = r.pipeline(transaction=True)
pipe.decrby('account:A:balance', 500)
pipe.incrby('account:B:balance', 500)
pipe.execute()   # atomic
```

---

In [5]:
import fakeredis, json, time

# fakeredis: full redis-py API, no server required
# Replace with redis.Redis(host='...') for a live server
r = fakeredis.FakeRedis(decode_responses=True)
print('Redis ready:', r.ping())


Redis ready: True


In [6]:
import fakeredis, json

r = fakeredis.FakeRedis(decode_responses=True)

print("=== MULTI/EXEC: atomic transaction block ===")
print()

print("MULTI/EXEC semantics:")
print("  - MULTI: opens a transaction queue")
print("  - Commands after MULTI are QUEUED, not executed")
print("  - EXEC: executes all queued commands atomically")
print("  - DISCARD: abandon the queued commands")
print()

# Setup
r.set('account:A001:balance', '5000')
r.set('account:A002:balance', '1200')

# Transfer $500 from A001 to A002 using MULTI/EXEC
pipe = r.pipeline(transaction=True)   # pipeline(transaction=True) = MULTI/EXEC
pipe.decrby('account:A001:balance', 500)
pipe.incrby('account:A002:balance', 500)
pipe.set('transfer:log:001', json.dumps({'from':'A001','to':'A002','amount':500}))
results = pipe.execute()

print(f"Transfer results: {results}")
print(f"A001 balance: {r.get('account:A001:balance')}")
print(f"A002 balance: {r.get('account:A002:balance')}")

print()
# MULTI/EXEC properties
properties = [
    ("Atomicity",    "All commands execute, or none do (on EXEC)"),
    ("Isolation",    "No other client sees partial state between MULTI and EXEC"),
    ("No rollback",  "EXEC always runs all commands; errors in individual commands do not abort"),
    ("Queue",        "Commands return QUEUED between MULTI and EXEC — no intermediate results"),
    ("DISCARD",      "Abandons all queued commands without executing"),
]
for name, desc in properties:
    print(f"  {name:<14s}  {desc}")

print()
print("IMPORTANT: MULTI/EXEC does NOT rollback on runtime error:")
print("""
  MULTI
  SET balance 1000      -- succeeds
  INCR balance_string   -- fails (not an integer) — but...
  SET other_key 'ok'    -- STILL executes
  EXEC
  -- Result: balance=1000, other_key='ok', error on INCR
  -- Redis does NOT roll back the successful commands
  -- Use WATCH for optimistic locking (see next section)
""")


=== MULTI/EXEC: atomic transaction block ===

MULTI/EXEC semantics:
  - MULTI: opens a transaction queue
  - Commands after MULTI are QUEUED, not executed
  - EXEC: executes all queued commands atomically
  - DISCARD: abandon the queued commands

Transfer results: [4500, 1700, True]
A001 balance: 4500
A002 balance: 1700

  Atomicity       All commands execute, or none do (on EXEC)
  Isolation       No other client sees partial state between MULTI and EXEC
  No rollback     EXEC always runs all commands; errors in individual commands do not abort
  Queue           Commands return QUEUED between MULTI and EXEC — no intermediate results
  DISCARD         Abandons all queued commands without executing

IMPORTANT: MULTI/EXEC does NOT rollback on runtime error:

  MULTI
  SET balance 1000      -- succeeds
  INCR balance_string   -- fails (not an integer) — but...
  SET other_key 'ok'    -- STILL executes
  EXEC
  -- Result: balance=1000, other_key='ok', error on INCR
  -- Redis does NOT roll

---
## WATCH: optimistic locking

In [7]:
import threading, time

print("=== WATCH: optimistic locking ===")
print()

r.set('account:A001:balance', '5000')

print("WATCH pattern (optimistic concurrency control):")
print("  1. WATCH key(s) — mark key(s) to monitor")
print("  2. Read the current value(s)")
print("  3. MULTI — open transaction")
print("  4. Queue commands based on read values")
print("  5. EXEC — if any watched key changed since WATCH, EXEC returns None (abort)")
print()

def transfer_with_watch(from_acc, to_acc, amount, max_retries=3):
    for attempt in range(1, max_retries + 1):
        with r.pipeline() as pipe:
            try:
                pipe.watch(f'account:{from_acc}:balance')
                balance = int(pipe.get(f'account:{from_acc}:balance') or 0)
                if balance < amount:
                    return False, f"Insufficient funds: {balance} < {amount}"
                pipe.multi()   # switch pipeline to MULTI mode
                pipe.decrby(f'account:{from_acc}:balance', amount)
                pipe.incrby(f'account:{to_acc}:balance', amount)
                pipe.execute()  # returns None if WATCH key was modified
                return True, f"Transfer OK on attempt {attempt}"
            except Exception as e:
                # EXEC returned None (watched key changed) or other error
                continue  # retry
    return False, "Max retries exceeded"

ok, msg = transfer_with_watch('A001', 'A002', 200)
print(f"Transfer result: {ok}  msg={msg!r}")
print(f"A001 balance: {r.get('account:A001:balance')}")
print(f"A002 balance: {r.get('account:A002:balance')}")

print()
print("WATCH vs MULTI/EXEC:")
comparison = [
    ("MULTI/EXEC",         "Atomic queue — no concurrent check; last-write-wins if two clients race"),
    ("WATCH + MULTI/EXEC", "Optimistic lock — detects concurrent modification; retries on conflict"),
    ("Use MULTI/EXEC when","Only one client modifies the key (no concurrent writes expected)"),
    ("Use WATCH when",     "Multiple clients may modify the same key (transfer, inventory decrement)"),
]
for name, desc in comparison:
    print(f"  {name:<26s}  {desc}")


=== WATCH: optimistic locking ===

WATCH pattern (optimistic concurrency control):
  1. WATCH key(s) — mark key(s) to monitor
  2. Read the current value(s)
  3. MULTI — open transaction
  4. Queue commands based on read values
  5. EXEC — if any watched key changed since WATCH, EXEC returns None (abort)

Transfer result: True  msg='Transfer OK on attempt 1'
A001 balance: 4800
A002 balance: 1900

WATCH vs MULTI/EXEC:
  MULTI/EXEC                  Atomic queue — no concurrent check; last-write-wins if two clients race
  WATCH + MULTI/EXEC          Optimistic lock — detects concurrent modification; retries on conflict
  Use MULTI/EXEC when         Only one client modifies the key (no concurrent writes expected)
  Use WATCH when              Multiple clients may modify the same key (transfer, inventory decrement)


---
## Lua scripts: atomic read-modify-write

In [10]:
print("=== Lua scripts: complex atomic operations ===")
print()

print("Why Lua scripts?")
print("  - MULTI/EXEC cannot read results mid-transaction")
print("  - Lua scripts execute atomically AND can use intermediate results")
print("  - Scripts are cached on the server (EVALSHA for repeated calls)")
print()

# Example: atomic rate limiter using Lua
# Allows max N requests per window_seconds per key
rate_limit_lua = """
local key     = KEYS[1]
local limit   = tonumber(ARGV[1])
local window  = tonumber(ARGV[2])
local current = redis.call('INCR', key)
if current == 1 then
    redis.call('EXPIRE', key, window)
end
if current > limit then
    return 0  -- rate limited
else
    return 1  -- allowed
end
"""
print("Rate limiter Lua script (production):")
print(rate_limit_lua)

# fakeredis does not support EVAL/EVALSHA/SCRIPT LOAD.
# The function below replicates the same logic using plain Redis commands
# so the demo output is accurate. In production, replace this with:
#   sha = r.script_load(rate_limit_lua)
#   allowed = r.evalsha(sha, 1, key, limit, window)
def rate_limit(r, key, limit, window):
    """Simulate the Lua rate limiter using individual Redis calls.
    Not atomic — for demo/testing only. Use Lua in production."""
    current = r.incr(key)
    if current == 1:
        r.expire(key, window)
    return 0 if current > limit else 1

# Test: 6 requests with limit=3 per 60s window
r_limit_key = 'ratelimit:patient:P001:/api/labs'
print(f"Testing rate limit: max=3 per 60s window on key {r_limit_key!r}")
print("(Simulated via plain Redis commands — swap for r.evalsha() in production)\n")
for i in range(1, 7):
    allowed = rate_limit(r, r_limit_key, 3, 60)
    status = 'ALLOWED' if allowed else 'RATE LIMITED'
    count  = r.get(r_limit_key)
    print(f"  Request {i}: {status:<14s}  count={count}")

print()
print("Other Lua use cases:")
lua_uses = [
    ("Atomic check-and-set",      "Read value, compute new value, SET — all in one script"),
    ("Atomic decrement with min",  "DECRBY but stop at 0 (inventory: never go negative)"),
    ("Batch conditional ops",      "Multiple HSETNX calls with shared condition"),
    ("Idempotent writes",          "Check if operation already done before executing"),
]
for name, desc in lua_uses:
    print(f"  {name:<26s}  {desc}")

=== Lua scripts: complex atomic operations ===

Why Lua scripts?
  - MULTI/EXEC cannot read results mid-transaction
  - Lua scripts execute atomically AND can use intermediate results
  - Scripts are cached on the server (EVALSHA for repeated calls)

Rate limiter Lua script (production):

local key     = KEYS[1]
local limit   = tonumber(ARGV[1])
local window  = tonumber(ARGV[2])
local current = redis.call('INCR', key)
if current == 1 then
    redis.call('EXPIRE', key, window)
end
if current > limit then
    return 0  -- rate limited
else
    return 1  -- allowed
end

Testing rate limit: max=3 per 60s window on key 'ratelimit:patient:P001:/api/labs'
(Simulated via plain Redis commands — swap for r.evalsha() in production)

  Request 1: ALLOWED         count=1
  Request 2: ALLOWED         count=2
  Request 3: ALLOWED         count=3
  Request 4: RATE LIMITED    count=4
  Request 5: RATE LIMITED    count=5
  Request 6: RATE LIMITED    count=6

Other Lua use cases:
  Atomic check-and-set  

---
## Common Pitfalls

**1. Expecting MULTI/EXEC to rollback on runtime errors**
MULTI/EXEC does NOT rollback. If one command fails (e.g., `INCR` on a string), the other commands still execute. This is by design — Redis prioritises throughput. Use WATCH for conditional execution, not MULTI alone for all-or-nothing guarantees.

**2. Reading values inside MULTI**
Commands between MULTI and EXEC return `QUEUED`, not their values. You cannot read a value, compute on it, and use the result in the same MULTI block. For read-modify-write atomically, use WATCH + MULTI/EXEC or a Lua script.

**3. Not handling WatchError in the retry loop**
When WATCH detects a concurrent modification, `pipe.execute()` raises `WatchError`. The retry loop must catch this, re-read the current value, and re-queue commands. Without proper retry logic, the transaction silently fails.

**4. Infinite retry loops with WATCH**
Under high concurrency, WATCH conflicts may repeat indefinitely. Always cap retries (`for attempt in range(max_retries)`) and raise an error or return a failure status when retries are exhausted.

**5. Using Lua KEYS and ARGV incorrectly**
In Lua, `KEYS` and `ARGV` are 1-indexed Lua tables. `KEYS[1]` is the first key, `ARGV[1]` is the first argument. `KEYS[0]` is nil (not an error, but wrong). Passing `numkeys=0` to `EVAL` when your script uses `KEYS` causes the script to see no keys and fail silently.

**6. Lua scripts blocking Redis**
Redis executes Lua scripts atomically on a single thread. A long-running or infinite-loop Lua script blocks ALL other Redis commands until it completes. Keep Lua scripts short, fast, and side-effect-free. Use `redis.call()` inside Lua (raises an error on failure); use `redis.pcall()` to catch errors without aborting the script.


---
*sql_methods_library - Samantha McGarrigle*